In [1]:
# python version
import sys
print(sys.executable)

C:\Users\felip\AppData\Local\Programs\Python\Python312\python.exe


In [2]:
import sys
import os
import pandas as pd
import numpy as np
import torch
import time

sys.path.append(os.path.abspath('../src'))

from sklearn.model_selection import train_test_split
from sklearn.base import clone

In [3]:
# device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [4]:
n_threads = os.cpu_count()
torch.set_num_threads(n_threads)
n_threads

8

In [5]:
train = pd.read_csv("../data/train.csv")

In [6]:
import re

TOKEN_PATTERN = re.compile(r"\w+(?:['-]\w+)*|[^\w\s]", flags=re.UNICODE)

def tokenize(text):
    return TOKEN_PATTERN.findall(text)

In [42]:
train['has_apos'] = train['english'].str.contains("'")
train['tokenized'] = train['english'].apply(tokenize)
sentences = train.iloc[:, 0].apply(tokenize)
sentences.values

array([list(['Two', 'young', ',', 'White', 'males', 'are', 'outside', 'near', 'many', 'bushes', '.']),
       list(['Several', 'men', 'in', 'hard', 'hats', 'are', 'operating', 'a', 'giant', 'pulley', 'system', '.']),
       list(['A', 'little', 'girl', 'climbing', 'into', 'a', 'wooden', 'playhouse', '.']),
       ...,
       list(['Two', 'male', 'construction', 'workers', 'are', 'working', 'on', 'a', 'street', 'outside', "someone's", 'home']),
       list(['An', 'elderly', 'man', 'sits', 'outside', 'a', 'storefront', 'accompanied', 'by', 'a', 'young', 'boy', 'with', 'a', 'cart', '.']),
       list(['A', 'man', 'in', 'shorts', 'and', 'a', 'Hawaiian', 'shirt', 'leans', 'over', 'the', 'rail', 'of', 'a', 'pilot', 'boat', ',', 'with', 'fog', 'and', 'mountains', 'in', 'the', 'background', '.'])],
      shape=(29000,), dtype=object)

In [65]:
from collections import Counter

counter = Counter(
    token
    for sentence in sentences
    for token in sentence
)
counter = dict(sorted(counter.items()))

min_freq = 2

vocab = list(
    token
    for token, freq in counter.items()
    if freq >= min_freq
)

special_characters = ["PAD", "UNK", "BOS", "EOS"]

word2idx = {word: idx for idx, word in enumerate(special_characters + vocab)}
idx2word = {idx: word for word, idx in word2idx.items()}

print(len(counter), idx2word)

11381 {0: 'PAD', 1: 'UNK', 2: 'BOS', 3: 'EOS', 4: '!', 5: '"', 6: '#', 7: '%', 8: '&', 9: "'", 10: '(', 11: ')', 12: ',', 13: '-', 14: '.', 15: '000', 16: '1', 17: '10', 18: '11', 19: '12', 20: '13', 21: '14', 22: '2', 23: '2008', 24: '2012', 25: '24', 26: '3', 27: '30', 28: '38', 29: '3rd', 30: '4', 31: '5', 32: '50', 33: '6', 34: '63', 35: '7', 36: '8', 37: "80's", 38: '9', 39: ':', 40: ';', 41: '?', 42: 'A', 43: 'ATM', 44: 'ATV', 45: "ATV's", 46: 'About', 47: 'Adult', 48: 'Adults', 49: 'Africa', 50: 'African', 51: 'African-American', 52: 'After', 53: 'Airline', 54: 'Airways', 55: 'Alaska', 56: 'All', 57: 'Am', 58: 'America', 59: "America's", 60: 'American', 61: 'Americans', 62: 'Amish', 63: 'An', 64: 'Angeles', 65: 'Angels', 66: 'Apple', 67: 'Arab', 68: 'Arabic', 69: 'Army', 70: 'Artist', 71: 'As', 72: 'Asia', 73: 'Asian', 74: 'Asians', 75: 'At', 76: 'Athlete', 77: 'Athletes', 78: 'Atlanta', 79: 'Attractive', 80: 'Autumn', 81: 'B', 82: 'BMW', 83: 'BMX', 84: 'Baby', 85: 'Bald', 86: '